# ITT (Indice de Transformacion Territorial) - Barrio Obrero (V1, 16 manzanas) (Cali)

ITT con las **5 dimensiones reales** calculadas desde los datos de la zona
(Comuna 9), replicando el estandar de Ciudad Paraiso. Metodologia homologada
al proyecto (notebook 02): 5 dimensiones con **peso igual (20% cada una)**,
normalizacion **relativa a la zona** (eventos negativos invierten; positivos
directos), `ITT = media(5 dimensiones) * 100`. 4 niveles oficiales.

| Dimension | Variables | Direccion |
|---|---|---|
| Seguridad | hurtos + homicidios + VIF | inversa |
| Convivencia | rinas + SPA | inversa |
| Movilidad | siniestros viales | inversa |
| Entorno Urbano | censo arboreo | directa |
| Desarrollo Economico | negocios + empleo | directa |

> El indice de caminabilidad NO entra al ITT. La red peatonal/intersecciones
> OSM se incluyen solo como capa de contexto visual.

Niveles: Activacion 0-40 (#D55E00), Consolidacion 40-60 (#E69F00),
Transformacion 60-80 (#009E73), Escala 80-100 (#0072B2).
Salida: `mapa_html/Barrio_obrero/mapa_itt_barrio_obrero_v1.html`.


## 0. Setup Colab (opcional)

In [ ]:
def _en_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False
if _en_colab():
    import os,subprocess,sys
    REPO='https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git'
    DEST='/content/mapa_Analitico_ciudad_Paraiso'
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','pyarrow','openpyxl'],check=False)
    if not os.path.isdir(DEST): subprocess.run(['git','clone','--depth','1',REPO,DEST],check=False)
    if os.path.isdir(os.path.join(DEST,'notebooks')): os.chdir(os.path.join(DEST,'notebooks'))
    print('Colab listo:',os.getcwd())
else:
    print('Entorno local (uv). Setup Colab omitido.')


## 1. Calculo del ITT (5 dimensiones, cruce espacial por manzana)

In [ ]:
import os
ZONA_DIR="v1_barrio_obrero"
IPM_REL=os.path.join(*['IPM_BARRIO_OBRERO_2_POLIGONOS', 'IPM_Barrio_Obrero_V1.geojson'])
POLY_REL="poligono_Barrio_Obrero_V1.geojson"
SUB="(V1, 16 manzanas)"
ZOOM=16
import os, json
import numpy as np, pandas as pd, geopandas as gpd
# Paleta / niveles ITT.
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"; OKI_BERMELL="#D55E00"
OKI_GRIS="#333333"; BG="#F4F6F9"; CRS_WEB=4326; CRS_M=3116; LIMITE="#1B4F8A"
ITT_NIVELES=[(0.0,40.0,"Activacion","#D55E00"),(40.0,60.0,"Consolidacion","#E69F00"),
             (60.0,80.0,"Transformacion","#009E73"),(80.0,100.0,"Escala","#0072B2")]
def _itt_nivel(v):
    v=float(v)
    for lo,hi,nom,col in ITT_NIVELES:
        if v<=hi: return nom,col
    return ITT_NIVELES[-1][2],ITT_NIVELES[-1][3]

def _ruta(*cands):
    for p in cands:
        if os.path.isfile(p) or os.path.isdir(p): return os.path.abspath(p)
    return None
def _en(dirbase, pref):
    for f in os.listdir(dirbase):
        if f.startswith(pref) and f.endswith(".geojson"): return os.path.join(dirbase, f)
    return None

# Resolucion de rutas (local / Colab).
DATADIR=_ruta(os.path.join("..","data","Barrio_Obrero_geojson",ZONA_DIR),
              os.path.join("data","Barrio_Obrero_geojson",ZONA_DIR),
              os.path.join("/content","data","Barrio_Obrero_geojson",ZONA_DIR))
IPM_PATH=_ruta(os.path.join("..","data","Barrio_Obrero_geojson",IPM_REL),
               os.path.join("data","Barrio_Obrero_geojson",IPM_REL),
               os.path.join("/content","data","Barrio_Obrero_geojson",IPM_REL))
if DATADIR is None or IPM_PATH is None:
    raise FileNotFoundError("Faltan datos de Barrio Obrero "+SUB)

# Manzanas base (del IPM, trae la geometria por manzana y el COD_MZN).
mz=gpd.read_file(IPM_PATH).to_crs(CRS_M)
mz=mz[mz.geometry.notna()].reset_index(drop=True)
mz["mid"]=range(len(mz))
mz["area_m2"]=mz.geometry.area.values

def norm_inv(s):
    s=s.astype(float); rng=s.max()-s.min()
    return pd.Series(1.0,index=s.index) if rng==0 else 1.0-(s-s.min())/rng
def norm_dir(s):
    s=s.astype(float); rng=s.max()-s.min()
    return pd.Series(0.5,index=s.index) if rng==0 else (s-s.min())/rng

# Conteo de eventos por manzana (sjoin within).
EVT={"homic":"02_seguridad_homicidios","hurtos":"03_seguridad_hurtos",
     "rinas":"04_convivencia_rinas","spa":"05_convivencia_spa","vif":"06_cohesion_vif",
     "siniestros":"07_movilidad_siniestros","arboles":"08_entorno_censo_arboreo",
     "negocios":"09_desarrollo_economico_negocios"}
C={}
for k,pref in EVT.items():
    p=_en(DATADIR,pref)
    if p is None: C[k]=pd.Series(0,index=range(len(mz))); continue
    g=gpd.read_file(p).to_crs(CRS_M)
    g=g[g.geometry.notna() & g.geom_type.str.contains("Point")]
    if len(g)==0: C[k]=pd.Series(0,index=range(len(mz))); continue
    j=gpd.sjoin(g[["geometry"]],mz[["mid","geometry"]],predicate="within",how="inner")
    C[k]=j.groupby("mid").size().reindex(range(len(mz)),fill_value=0)
C=pd.DataFrame(C)
# Empleo por manzana (personal ocupado de negocios).
pneg=_en(DATADIR,"09_desarrollo_economico_negocios")
if pneg:
    gneg=gpd.read_file(pneg).to_crs(CRS_M)
    _ec=[c for c in ("personal_o","empleo_personal_ocupado","personal_ocupado") if c in gneg.columns]
    gneg["_emp"]=pd.to_numeric(gneg[_ec[0]],errors="coerce").fillna(0) if _ec else 0
    jn=gpd.sjoin(gneg[["_emp","geometry"]],mz[["mid","geometry"]],predicate="within",how="inner")
    C["empleo"]=jn.groupby("mid")["_emp"].sum().reindex(range(len(mz)),fill_value=0)
else:
    C["empleo"]=0

# Scores por dimension (0-1), ITT = media*100.
mz["d_seguridad"]=(norm_inv(C["hurtos"]+C["homic"]+C["vif"])*100).round(1).values
mz["d_convivencia"]=(norm_inv(C["rinas"]+C["spa"])*100).round(1).values
mz["d_movilidad"]=(norm_inv(C["siniestros"])*100).round(1).values
mz["d_entorno"]=(norm_dir(C["arboles"])*100).round(1).values
mz["d_economico"]=(norm_dir(C["negocios"]+C["empleo"])*100).round(1).values
mz["itt"]=((mz["d_seguridad"]+mz["d_convivencia"]+mz["d_movilidad"]+mz["d_entorno"]+mz["d_economico"])/5.0).round(1)
# Conteos crudos (para popups/panel).
for k in C.columns: mz["n_"+k]=C[k].values

# Resumen de zona (promedio de dimensiones y del ITT).
DIMS=[("Seguridad","d_seguridad"),("Convivencia","d_convivencia"),("Movilidad","d_movilidad"),
      ("Entorno Urbano","d_entorno"),("Desarrollo Economico","d_economico")]
ITT_ZONA=round(float(mz["itt"].mean()),1)
DIM_ZONA={nom:round(float(mz[col].mean()),1) for nom,col in DIMS}
_niv_zona,_=_itt_nivel(ITT_ZONA)
mz=mz.to_crs(CRS_WEB)
POLIGONO_ZONA=gpd.read_file(IPM_PATH).to_crs(CRS_WEB)  # placeholder, se reemplaza abajo
POLY_PATH=_ruta(os.path.join("..","data","Barrio_Obrero_geojson",POLY_REL),
                os.path.join("data","Barrio_Obrero_geojson",POLY_REL),
                os.path.join("/content","data","Barrio_Obrero_geojson",POLY_REL))
POLIGONO_ZONA=gpd.read_file(POLY_PATH).to_crs(CRS_WEB)
_b=POLIGONO_ZONA.total_bounds; BO_BOUNDS=[[float(_b[1]),float(_b[0])],[float(_b[3]),float(_b[2])]]
OUT_DIR=None
for _d in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try: os.makedirs(_d,exist_ok=True); OUT_DIR=os.path.abspath(_d); break
    except Exception: pass
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html","Barrio_obrero"))
os.makedirs(MAPAS_DIR,exist_ok=True)
print("Manzanas:",len(mz),"| ITT zona: %.1f -> %s"%(ITT_ZONA,_niv_zona))
for nom,col in DIMS: print("  %-22s %.1f"%(nom,DIM_ZONA[nom]))
print("Salida:",MAPAS_DIR)

## 2. Tabla resumen del ITT por dimension y consolidado

In [ ]:
_tab=pd.DataFrame({"Dimension":[n for n,_ in DIMS]+["ITT CONSOLIDADO"],
  "Score (0-100)":[DIM_ZONA[n] for n,_ in DIMS]+[ITT_ZONA],
  "Nivel":[_itt_nivel(DIM_ZONA[n])[0] for n,_ in DIMS]+[_itt_nivel(ITT_ZONA)[0]]})
print(_tab.to_string(index=False))

## 3. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 4. Construccion del mapa (Folium) - estandar Ciudad Paraiso

In [ ]:
ZOOM=16
import folium
_c=POLIGONO_ZONA.geometry.union_all().centroid
m=folium.Map(location=[_c.y,_c.x],zoom_start=ZOOM,tiles=None,control_scale=True,
             zoom_snap=0.25,zoom_delta=0.1,max_zoom=22)
m.get_root().width="100%"; m.get_root().height="100%"
m.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}"
    ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}"
    ".leaflet-top,.leaflet-bottom{z-index:1000}"
    "#itt-panel{z-index:1001;background:rgba(255,255,255,.95);backdrop-filter:blur(4px);"
    "-webkit-backdrop-filter:blur(4px);box-shadow:0 4px 15px rgba(0,0,0,.2)}"
    ".export-ctl{z-index:1002}</style>"))
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",attr="Google",name="Google Streets",show=True,max_zoom=22,max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",attr="Esri",name="Esri Satelite",show=False,max_zoom=22,max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",attr="Google",name="Google Satelite",show=False,max_zoom=22,max_native_zoom=20).add_to(m)
# Mascara.
_zg=POLIGONO_ZONA.geometry.union_all(); _mundo=[[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]; _ag=[]
for _gm in (list(_zg.geoms) if _zg.geom_type=="MultiPolygon" else [_zg]):
    _ag.append([[x,y] for x,y in _gm.exterior.coords])
folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},
    name="Mascara (foco en la zona)",control=False,
    style_function=lambda f:{"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.18}).add_to(m)

# 1) Coropleta ITT por manzana (score consolidado, 5 dimensiones). Activa.
def _popup(r):
    v=float(r["itt"]); nom,_=_itt_nivel(v)
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;background:%s;padding:7px 9px;border-radius:6px;min-width:230px">'
            '<div style="font-weight:700;color:%s">ITT manzana: %.1f / 100 &rarr; %s</div>'
            '<div style="margin-top:4px;font-size:11px">Dimensiones (0-100):</div>'
            '<div style="font-size:11px">&bull; Seguridad: <b>%.0f</b> (hurtos %d, homic %d, VIF %d)</div>'
            '<div style="font-size:11px">&bull; Convivencia: <b>%.0f</b> (rinas %d, SPA %d)</div>'
            '<div style="font-size:11px">&bull; Movilidad: <b>%.0f</b> (siniestros %d)</div>'
            '<div style="font-size:11px">&bull; Entorno: <b>%.0f</b> (arboles %d)</div>'
            '<div style="font-size:11px">&bull; Desarrollo eco: <b>%.0f</b> (negocios %d, empleo %d)</div>'
            '<div style="font-size:9px;color:#888;margin-top:4px">ITT = media de 5 dimensiones (peso igual). Norm. relativa a la zona.</div></div>'
            % (BG,LIMITE,v,nom,r["d_seguridad"],int(r["n_hurtos"]),int(r["n_homic"]),int(r["n_vif"]),
               r["d_convivencia"],int(r["n_rinas"]),int(r["n_spa"]),r["d_movilidad"],int(r["n_siniestros"]),
               r["d_entorno"],int(r["n_arboles"]),r["d_economico"],int(r["n_negocios"]),int(r["n_empleo"])))
_fg=folium.FeatureGroup(name="ITT: manzanas (indice consolidado)",show=True)
for _,r in mz.iterrows():
    nom,col=_itt_nivel(float(r["itt"]))
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=(lambda f,c=col:{"fillColor":c,"color":"#ffffff","weight":0.6,"fillOpacity":0.82}),
        tooltip="ITT %.1f (%s)"%(float(r["itt"]),nom),popup=folium.Popup(_popup(r),max_width=300)).add_to(_fg)
_fg.add_to(m)

# 2) Una coropleta por dimension (apagadas), para explorar cada eje.
_DIMCOL=[("Seguridad","d_seguridad"),("Convivencia","d_convivencia"),("Movilidad","d_movilidad"),
         ("Entorno Urbano","d_entorno"),("Desarrollo Economico","d_economico")]
for nomd,cold in _DIMCOL:
    fgd=folium.FeatureGroup(name="Dimension: "+nomd,show=False)
    for _,r in mz.iterrows():
        _n,_col=_itt_nivel(float(r[cold]))
        folium.GeoJson(r["geometry"].__geo_interface__,
            style_function=(lambda f,c=_col:{"fillColor":c,"color":"#fff","weight":0.4,"fillOpacity":0.72}),
            tooltip="%s: %.0f"%(nomd,float(r[cold]))).add_to(fgd)
    fgd.add_to(m)

# 3) Capas de eventos puntuales (contexto, apagadas). Colores Okabe-Ito por tema.
import os as _os
_EVT_CAPAS=[("Seguridad: hurtos","03_seguridad_hurtos","#0072B2"),
            ("Seguridad: homicidios","02_seguridad_homicidios","#D55E00"),
            ("Seguridad: VIF","06_cohesion_vif","#CC79A7"),
            ("Convivencia: rinas","04_convivencia_rinas","#009E73"),
            ("Convivencia: SPA","05_convivencia_spa","#E69F00"),
            ("Movilidad: siniestros","07_movilidad_siniestros","#56B4E9"),
            ("Entorno: arbolado","08_entorno_censo_arboreo","#009E73"),
            ("Desarrollo eco: negocios","09_desarrollo_economico_negocios","#0072B2")]
for nomc,pref,colr in _EVT_CAPAS:
    pth=_en(DATADIR,pref)
    if pth is None: continue
    gg=gpd.read_file(pth).to_crs(CRS_WEB)
    gg=gg[gg.geometry.notna() & gg.geom_type.str.contains("Point")]
    fgp=folium.FeatureGroup(name=nomc+" ("+str(len(gg))+")",show=False)
    for _,rr in gg.iterrows():
        folium.CircleMarker([rr.geometry.y,rr.geometry.x],radius=3,color=colr,fill=True,
            fill_color=colr,fill_opacity=0.75,weight=0.5).add_to(fgp)
    fgp.add_to(m)

# 4) Red peatonal OSM (contexto movilidad, lineas).
_prp=_en(DATADIR,"11_red_peatonal_osm")
if _prp:
    gr=gpd.read_file(_prp).to_crs(CRS_WEB)
    folium.GeoJson(gr.to_json(),name="Movilidad: red peatonal OSM",show=False,
        style_function=lambda f:{"color":"#888","weight":1.2,"opacity":0.7}).add_to(m)

# 5) Limite perimetral (azul, fijo y visible).
folium.GeoJson(POLIGONO_ZONA.to_json(),name="Limite Barrio Obrero",show=True,
    style_function=lambda f:{"color":LIMITE,"weight":2.5,"fill":False}).add_to(m)

# Leyenda de niveles ITT.
_items=""
for lo,hi,nom,col in ITT_NIVELES:
    _items+=('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
             '<i style="width:16px;height:12px;background:%s;display:inline-block;border:1px solid #999"></i>'
             '<span style="font-size:10px">%s (%d-%d)</span></div>'%(col,nom,int(lo),int(hi)))
_ley=('<div id="itt-leyenda" style="position:fixed;bottom:24px;right:24px;z-index:1000;background:'+BG+';'
      'padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);font-family:system-ui,Segoe UI,sans-serif;color:'+OKI_GRIS+';max-width:220px">'
      '<div style="font-weight:700;font-size:11px;margin-bottom:4px">Nivel ITT</div>'+_items+
      '<div style="font-size:9px;color:#888;margin-top:4px">ITT = media de 5 dimensiones. Normalizacion relativa a la zona.</div></div>')
m.get_root().html.add_child(folium.Element(_ley))
m.fit_bounds(BO_BOUNDS,padding=(30,30))
folium.LayerControl(collapsed=True,position="topright").add_to(m)
print("Mapa ITT real construido. Manzanas:",len(mz))

## 5. Widget flotante: ITT consolidado + 5 dimensiones

In [ ]:
SUB="(V1)"
_cont={nom:0 for _lo,_hi,nom,_c in ITT_NIVELES}
for _v in mz["itt"]:
    _n,_=_itt_nivel(_v); _cont[_n]+=1
_dim_labels=[nom for nom,_ in DIMS]
_dim_vals=[DIM_ZONA[nom] for nom,_ in DIMS]
_chart=json.dumps({"labels":_dim_labels,"data":_dim_vals},ensure_ascii=False)
_nz,_=_itt_nivel(ITT_ZONA)
_resumen=('<div style="margin:2px 0 6px 0">ITT de zona: <b style="font-size:15px;color:%s">%.1f</b> / 100 &nbsp;&rarr;&nbsp; <b>%s</b></div>'%(LIMITE,ITT_ZONA,_nz))
_resumen+='<div style="margin:4px 0 2px;font-weight:700;font-size:11px">Manzanas por nivel ITT:</div>'
for _lo,_hi,nom,col in ITT_NIVELES:
    _resumen+=('<div style="display:flex;align-items:center;gap:6px;margin:1px 0;font-size:11px">'
               '<i style="width:12px;height:12px;background:%s;display:inline-block;border-radius:2px;border:1px solid #777"></i>'
               '<span>%s: <b>%d</b></span></div>'%(col,nom,_cont[nom]))
_panel = r"""
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<div id="itt-panel" style="position:fixed;bottom:25px;left:20px;z-index:1001;max-width:400px;width:min(400px,44vw);max-height:84vh;overflow-y:auto;padding:11px 13px;border-radius:10px;font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="display:flex;align-items:center;justify-content:space-between;gap:8px">
    <div style="font-weight:700;font-size:13px">ITT - Barrio Obrero __SUB__</div>
    <button id="itt-toggle" title="Minimizar / expandir" style="cursor:pointer;border:1px solid #bbb;background:#fff;color:#333;width:24px;height:22px;border-radius:5px;font-size:13px;line-height:1;font-weight:700;flex:0 0 auto">_</button>
  </div>
  <div id="itt-body">
    <div style="font-size:10px;color:#666;margin:4px 0 6px 0">Indice de Transformacion Territorial. 5 dimensiones con peso igual (20%). Normalizacion relativa a la zona.</div>
    __RESUMEN__
    <details open style="margin-top:8px">
      <summary style="cursor:pointer;font-weight:600;color:__LIM__;outline:none">Dimensiones del ITT (promedio zona) / Ocultar</summary>
      <div style="position:relative;height:210px;width:100%;margin-top:6px"><canvas id="itt-chart"></canvas></div>
      <div style="font-size:9px;color:#777;margin-top:3px">Escala 0-100. Mas alto = mejor transformacion.</div>
    </details>
    <details style="margin-top:6px">
      <summary style="cursor:pointer;font-weight:600;color:#777;outline:none">Composicion de las dimensiones / Ver</summary>
      <div style="font-size:11px;color:#555;margin-top:6px;line-height:1.45">
        &bull; <b>Seguridad</b>: hurtos + homicidios + VIF (inversa)<br>
        &bull; <b>Convivencia</b>: rinas + SPA (inversa)<br>
        &bull; <b>Movilidad</b>: siniestros viales (inversa)<br>
        &bull; <b>Entorno Urbano</b>: censo arboreo (directa)<br>
        &bull; <b>Desarrollo Economico</b>: negocios + empleo (directa)
      </div>
    </details>
  </div>
</div>
<script>
(function(){
  var D=__CHART__;
  function col(v){ return v<40?"#D55E00":(v<60?"#E69F00":(v<80?"#009E73":"#0072B2")); }
  function dibujar(){
    var cv=document.getElementById("itt-chart"); if(!cv||typeof Chart==="undefined") return;
    new Chart(cv.getContext("2d"),{type:"bar",
      data:{labels:D.labels,datasets:[{data:D.data,backgroundColor:D.data.map(col),borderWidth:1}]},
      options:{indexAxis:"y",responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},
        scales:{x:{beginAtZero:true,max:100,ticks:{font:{size:9}},title:{display:true,text:"Score (0-100)",font:{size:9}}},y:{ticks:{font:{size:9}}}}}});
  }
  function initColapso(){
    var b=document.getElementById("itt-toggle"),bo=document.getElementById("itt-body");
    if(!b||!bo)return; b.addEventListener("click",function(){var o=(bo.style.display==="none");bo.style.display=o?"":"none";b.textContent=o?"_":"+";});
  }
  if(document.readyState==="complete"){setTimeout(dibujar,300);initColapso();}
  else{window.addEventListener("load",function(){setTimeout(dibujar,300);initColapso();});}
})();
</script>
"""
_panel=(_panel.replace("__GRIS__",OKI_GRIS).replace("__LIM__",LIMITE).replace("__SUB__",SUB)
             .replace("__RESUMEN__",_resumen).replace("__CHART__",_chart))
m.get_root().html.add_child(folium.Element(_panel))
print("Widget ITT inyectado.")

## 6. Exportacion del HTML

In [ ]:
agregar_export(m, "ITT - Barrio Obrero (V1, 16 manzanas)", "itt_bo_v1")
html=os.path.join(MAPAS_DIR,"mapa_itt_barrio_obrero_v1.html")
m.save(html)
print("Mapa ITT Barrio Obrero (V1, 16 manzanas) exportado:", html)
m